# Wireless Coverage — Part 2a: Surfaces from the Point Cloud (production DTM via TIN)

![Wireless Coverage Part 2a — LiDAR point cloud to DSM + TIN DTM + CHM permanent tables](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/wireless-coverage/wireless-coverage-02a.png)

**The main arc.** Part 1 turned the LiDAR point cloud into DSM, DTM, and CHM by binning. Part 2a
*builds on Part 1* — it reads Part 1's point cloud and DSM back (no re-reading the raw `.laz`) and
replaces the binned ground with a **Delaunay-TIN bare-earth DTM** (`rst_dtmfromgeoms_agg` over the
LiDAR ground returns): a continuous, gap-free terrain model, and the series' **production DTM**.

This is the "beyond pixels" path — the terrain comes from the *point cloud* (vector/point geometry),
not from a raster. We then difference the surfaces into CHM with `rst_chm` and write the canonical
`wc_surface_{dsm,dtm,chm}` tables **and** path-backed GeoTIFF tiles to `OUT_DIR` for Parts 3–4.

> **Alternative origin.** If you start from a **DSM raster** instead of a point cloud, see
> **Part 2b** (`02b_surfaces_from_dsm`): it derives DTM and CHM from the DSM alone and lands the same
> canonical surfaces, so Part 3 onward is identical either way.

> **Runtime.** Databricks Serverless environment 6 — the [lightweight tier](https://databrickslabs.github.io/geobrix/docs/api/execution-tiers) (pure Python/PySpark, no JAR or GDAL init script). Set `DEMO=False` to run the full San Francisco AOI.

---

**Last Update:** 2026-10-04

## Install GeoBrix

In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"

In [0]:
%restart_python

## Shared configuration

Shared imports, UDF registration, series parameters, and helpers are loaded from `config_nb` via `%run ./config_nb`. Notebook-specific overrides follow.

In [ ]:
%run ./config_nb

## Parameters

Shared series parameters (SF_AOI, CATALOG, SCHEMA, SRID, PIXEL_M, TILE_M, TPX, X0–Y1, OUT_DIR,
FORCE_RELOAD, DEMO) come from `config_nb`. Part 2a reads Part 1's tables (`SRC_PREFIX`) and writes
its own canonical surfaces (`TABLE_PREFIX`).

In [ ]:
# ── nb2a-specific overrides (after %run ./config_nb) ────────────────────────
# config_nb provides: F, rx, Window, _tbl, _persist, preview, plot_raster, plot_static,
#   lonlat_to_3857, X0, Y0, X1, Y1, SRID, PIXEL_M, TILE_M, TPX, OUT_DIR, CATALOG, SCHEMA.
SRC_PREFIX   = "wc_lidar"    # read Part 1's tables: wc_lidar_{pts,dsm}
TABLE_PREFIX = "wc_surface"  # write: CATALOG.SCHEMA.wc_surface_{dsm,dtm,chm}[_demo]
DEMO         = True          # True → Golden Gate Park ±2 tile window; False → full SF
FORCE_RELOAD = False         # True → rebuild the surface tables (else reuse)
TIN_MAX_PTS  = 150_000       # per-tile ground-return cap for the TIN (bounds triangulation memory)
# OUT_DIR (path-backed surfaces dir consumed by Part 4) comes from config_nb.
print(f"SRC_PREFIX={SRC_PREFIX} | TABLE_PREFIX={TABLE_PREFIX} | DEMO={DEMO}")

## Build on Part 1 — read the point cloud and DSM

Part 2a doesn't re-read the `.laz`. It reads the tables **Part 1** persisted — the classified point
cloud (`wc_lidar_pts`) for the ground returns the TIN needs, and the DSM (`wc_lidar_dsm`), which we
carry forward unchanged as the canonical surface. (Run `01_pure_lidar_chm` first, with the same
`DEMO` setting, so the `_demo` table suffixes line up.)

In [ ]:
# Read Part 1's persisted tables — no LAZ re-read (that would duplicate Part 1).
# _tbl appends the same _demo suffix as Part 1 when DEMO=True, so the tables line up.
_pts_tbl = _tbl(SRC_PREFIX, "pts")
_dsm_tbl = _tbl(SRC_PREFIX, "dsm")
for _t in (_pts_tbl, _dsm_tbl):
    assert spark.catalog.tableExists(_t), \
        f"{_t} not found — run 01_pure_lidar_chm first (same DEMO setting)."
pts = spark.table(_pts_tbl)
# Carry Part 1's DSM forward unchanged (max-z over all returns) as the canonical DSM surface.
dsm = _persist(spark.table(_dsm_tbl), "dsm")   # -> wc_surface_dsm
print(f"from Part 1: {pts.count():,} returns | DSM {dsm.count()} tiles -> {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_dsm")

## Bare-earth DTM — Delaunay TIN from the LiDAR ground returns (the production surface)

The binned DTM (Part 1) takes the `min` ground return per pixel and can leave gaps where no ground
return landed. Here we triangulate the ground returns into a **Delaunay TIN** and interpolate a
continuous, gap-free bare earth — the series' production DTM. Points are built product-natively with
`st_makepoint(x, y, z)` → WKB and fed to `rst_dtmfromgeoms_agg`, one TIN per tile.

A Delaunay TIN must hold all of a tile's points in memory to triangulate (unlike binning, whose
memory is bounded by the grid), so we cap each tile at `TIN_MAX_PTS` ground returns — a representative
subset yields a near-identical bare earth (standard LiDAR TIN practice) and keeps worker memory flat
as the AOI grows.

In [ ]:
# Ground returns (class 2) only; cap each TILE at TIN_MAX_PTS so every tile's triangulation
# fits in memory (the cap is per-tile, not global — a random bounded sample via row_number).
ground = (pts.where(F.col("classification") == 2)
             .select("tx", "ty", "t_xmin", "t_ymin", "t_xmax", "t_ymax", "x", "y", "z"))
_w = Window.partitionBy("tx", "ty").orderBy(F.rand(42))
ground_s = (ground.withColumn("_rn", F.row_number().over(_w))
                  .where(F.col("_rn") <= TIN_MAX_PTS)
                  .drop("_rn")
                  .withColumn("pt", F.expr("st_asbinary(st_makepoint(x, y, z))")))

# One Delaunay-TIN DTM tile per (tx, ty) via the streaming rst_dtmfromgeoms_agg aggregator.
dtm = ground_s.groupBy("tx", "ty").agg(
    rx.rst_dtmfromgeoms_agg("pt", F.lit(None), F.lit(0.0), F.lit(0.0),
                            "t_xmin", "t_ymin", "t_xmax", "t_ymax",
                            F.lit(TPX), F.lit(TPX), F.lit(SRID)).alias("dtm"))
dtm = _persist(dtm, "dtm")   # -> wc_surface_dtm (TIN bare earth, the production DTM)
print(f"TIN DTM {dtm.count()} tiles -> {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_dtm")

In [ ]:
# Preview the surface (DSM) vs the TIN bare earth (DTM) on a canopy-rich Golden Gate Park tile.
_gx, _gy = lonlat_to_3857(-122.486, 37.769)     # Golden Gate Park
_tx, _ty = int((_gx - X0) // TILE_M), int((_gy - Y0) // TILE_M)
_sel = (F.col("tx") == _tx) & (F.col("ty") == _ty)
dsm_t = dsm.where(_sel).first()
dtm_t = dtm.where(_sel).first()
print(f"sample tile tx={_tx} ty={_ty}")
if dsm_t: plot_raster(dsm_t["dsm"]["raster"], cmap="terrain", fig_w=7, fig_h=7)   # surface (DSM)
if dtm_t: plot_raster(dtm_t["dtm"]["raster"], cmap="terrain", fig_w=7, fig_h=7)   # TIN bare earth

## CHM — `rst_chm(DSM, DTM)`

Difference the DSM and the TIN bare-earth DTM with `rst_chm` (DSM − DTM, clamped ≥ 0) and persist the
canopy height model to `CATALOG.SCHEMA.wc_surface_chm`.

In [ ]:
chm = dsm.join(dtm, ["tx", "ty"]).select("tx", "ty", rx.rst_chm("dsm", "dtm").alias("chm"))
chm = _persist(chm, "chm")   # -> wc_surface_chm
print(f"{chm.count()} CHM tiles -> {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_chm")
chm_t = chm.where(_sel).first()
if chm_t:
    plot_raster(chm_t["chm"]["raster"], cmap="viridis", fig_w=7, fig_h=7)

## Write the canonical surfaces to the Volume

Persist DSM, the TIN bare-earth DTM, and CHM as path-backed GeoTIFF tiles under `OUT_DIR` (one
`gtiff_gbx` dir each). These are the **canonical surface products** the rest of the series reads:
Part 3 grids them to H3, and Part 4's per-tower viewsheds window the DSM tiles directly from these
files. Writing them here (not optionally) means a `01 → 02a` run leaves every downstream notebook a
complete, path-backed surface set.

In [ ]:
# One gtiff_gbx dir per surface: {OUT_DIR}/{dsm,dtm,chm}. The writer takes (source, tile) and
# emits <source>.tif; name each tile <kind>_<tx>_<ty> so Part 4 can recover (tx, ty) from the name.
for _df, _kind in [(dsm, "dsm"), (dtm, "dtm"), (chm, "chm")]:
    (_df.select(F.concat_ws("_", F.lit(_kind), "tx", "ty").alias("source"),
                F.col(_kind).alias("tile"))
        .write.format("gtiff_gbx").option("nameCol", "source").mode("overwrite").save(f"{OUT_DIR}/{_kind}"))
print("wrote canonical surfaces ->", OUT_DIR, "(dsm, dtm, chm)")

## Summary

Part 2a built on Part 1 — reading its point cloud and DSM back, with no `.laz` re-read — and produced
the series' **production surfaces**: DSM (carried forward), a **TIN bare-earth DTM**
(`rst_dtmfromgeoms_agg` over the LiDAR ground returns, the beyond-pixels terrain model), and CHM
(`rst_chm`). All three are written as canonical `wc_surface_{dsm,dtm,chm}` tables **and** path-backed
GeoTIFF tiles under `OUT_DIR`.

**Next:** Part 3 (`03_h3_gridding`) grids these surfaces to H3; Part 4 (`04_tower_viewsheds`) windows
the DSM tiles for per-tower line-of-sight. If you start from a DSM raster instead of a point cloud,
**Part 2b** (`02b_surfaces_from_dsm`) lands the same canonical surfaces first.